In [43]:
from pydantic import BaseModel,Field # returns Dictionary
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate # PromptTemplate --> Directly Prompt not like ChatPromptTemplate()
from langchain_core.output_parsers import PydanticOutputParser
import os

In [ ]:
'''
Why Pydantic,
For structring the output.
In real-time --> To validate the response or request
api payloads
'''

'\nWhy Pydantic,\nFor structring the output.\nIn real-time --> To validate the response or request\n'

In [45]:
with open('data.txt','r+') as file:
    data = file.read()

In [46]:
qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature=0
)
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

In [47]:
template = ''' 
You are a helpful assitant that can answer questions about the product.
Answer the question based on the product details provided.
product : {product}
question : {question}
'''

In [48]:
prompt_template = PromptTemplate.from_template(template)

In [49]:
# Field --> a tag for the data
class Product(BaseModel): #BaseModel --> Pydantic Model
    name :str = Field(description = "Name of the Product")
    price : str = Field(description="Price of the Product")
    description : str = Field(description="Description of the Product")
    category : str = Field(description="Category of the product")


In [50]:
qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature=0
).with_structured_output(Product)
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
).with_structured_output(Product)

In [51]:
# Create --> LCEL
chain = prompt_template | gemini_model

In [52]:
user_input = [
    {
        "product" : data,
        "question"  : "What is the price of the Iphone 15 Pro Max?"
    },
    {
        "product" :data,
        "question" : "What is the price of the galaxy S24 Ultra?"
    },
    {
        "product" : data,
        "question" : "What is the Price of Redmi?"
    },
    {
        "product" : data,
        "question" : "What is the price of the OnePlus?"
    }
]

In [53]:
user_input = (
    {
        "product" : data,
        "question"  : "What is the price of the White T-shirt?"
    })

In [ ]:
response = chain.invoke(user_input) # model_dump_json()--> Typecaste the output to json format

In [55]:
response.model_dump_json()

AttributeError: 'generator' object has no attribute 'model_dump_json'

In [ ]:
user_input = [
    {
        "product" : data,
        "question"  : "What is the price of the Iphone 15 Pro Max?"
    },
    {
        "product" :data,
        "question" : "What is the price of the galaxy S24 Ultra?"
    },
    {
        "product" : data,
        "question" : "What is the Price of Redmi?"
    },
    {
        "product" : data,
        "question" : "What is the price of the OnePlus?"
    }
]

In [ ]:
batch_response = chain.batch(user_input) 
# batch() --> Accepts multiple inputs
# invoke() --> For one input
# stream() --> For Quick Response

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 48m32.391670016s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Ge

In [ ]:
results = []
for result in batch_response:
    results.append(result.model_dump_json())
results

In [ ]:
results = []
for query in user_input:
    print(chain.invoke(query).model.dump())